Following the success of using structured output lets see if the tool calling functions from the langchain docs work.

Tools defined using the tool decorator.

In [2]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b


tools = [add, multiply]

Tools defined using pydantic.

In [6]:
from langchain_core.pydantic_v1 import BaseModel, Field


# Note that the docstrings here are crucial, as they will be passed along
# to the model along with the class name.
class Add(BaseModel):
    """Add two integers together."""

    a: int = Field(..., description="First integer")
    b: int = Field(..., description="Second integer")


class Multiply(BaseModel):
    """Multiply two integers together."""

    a: int = Field(..., description="First integer")
    b: int = Field(..., description="Second integer")


tools = [Add, Multiply]

Now proceed using tools with pydantic

In [7]:
from langchain_groq import ChatGroq
from typing import Optional

from langchain_core.pydantic_v1 import BaseModel, Field

llm = ChatGroq(model_name='llama3-70b-8192')
llm_with_tools = llm.bind_tools(tools)

In [8]:
query = "What is 3 * 12? Also, what is 11 + 49?"

llm_with_tools.invoke(query).tool_calls

[{'name': 'Multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_6e8c'},
 {'name': 'Add', 'args': {'a': 11, 'b': 49}, 'id': 'call_e328'}]

In [9]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser

chain = llm_with_tools | PydanticToolsParser(tools=[Multiply, Add])
chain.invoke(query)

[Multiply(a=3, b=12), Add(a=11, b=49)]

OK, I did not expect that to work. Keep going.

Streaming

In [10]:
async for chunk in llm_with_tools.astream(query):
    print(chunk.tool_call_chunks)

[{'name': 'Multiply', 'args': '{"a":3,"b":12}', 'id': 'call_6gpa', 'index': None}, {'name': 'Add', 'args': '{"a":11,"b":49}', 'id': 'call_nmah', 'index': None}]


OK - so we worked better than the example. continue on with their solution.

In [11]:
first = True
async for chunk in llm_with_tools.astream(query):
    if first:
        gathered = chunk
        first = False
    else:
        gathered = gathered + chunk

    print(gathered.tool_call_chunks)

[{'name': 'Multiply', 'args': '{"a":3,"b":12}', 'id': 'call_k93e', 'index': None}, {'name': 'Add', 'args': '{"a":11,"b":49}', 'id': 'call_b0r2', 'index': None}]


In [12]:
print(type(gathered.tool_call_chunks[0]["args"]))

<class 'str'>


Now lets pass tool outputs back to the model.

In [13]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage(query)]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply}[tool_call["name"].lower()]
    tool_output = selected_tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))
messages

[HumanMessage(content='What is 3 * 12? Also, what is 11 + 49?'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_d08a', 'function': {'arguments': '{"a":3,"b":12}', 'name': 'Multiply'}, 'type': 'function'}, {'id': 'call_ann3', 'function': {'arguments': '{"a":11,"b":49}', 'name': 'Add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 1076, 'total_tokens': 1168, 'completion_time': 0.254565709, 'prompt_time': 0.183067584, 'queue_time': None, 'total_time': 0.437633293}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_753a4aecf6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-50dad433-d45f-4456-bee9-555266b191d0-0', tool_calls=[{'name': 'Multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_d08a'}, {'name': 'Add', 'args': {'a': 11, 'b': 49}, 'id': 'call_ann3'}]),
 ToolMessage(content='36', tool_call_id='call_d08a'),
 ToolMessage(content='60', tool_call_id='call_ann3')]

In [14]:
llm_with_tools.invoke(messages)

AIMessage(content='The answer to 3 * 12 is 36 and the answer to 11 + 49 is 60.', response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 1198, 'total_tokens': 1222, 'completion_time': 0.064561352, 'prompt_time': 0.196590826, 'queue_time': None, 'total_time': 0.261152178}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_abd29e8833', 'finish_reason': 'stop', 'logprobs': None}, id='run-62fc553d-2f37-4ef2-86d3-5ba5ca687e3e-0')

Few shot prompting

In [15]:
llm_with_tools.invoke(
    "Whats 119 times 8 minus 20. Don't do any math yourself, only use tools for math. Respect order of operations"
).tool_calls

[]

Which is wrong. Try few shot prompting.

In [16]:
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

examples = [
    HumanMessage(
        "What's the product of 317253 and 128472 plus four", name="example_user"
    ),
    AIMessage(
        "",
        name="example_assistant",
        tool_calls=[
            {"name": "Multiply", "args": {"x": 317253, "y": 128472}, "id": "1"}
        ],
    ),
    ToolMessage("16505054784", tool_call_id="1"),
    AIMessage(
        "",
        name="example_assistant",
        tool_calls=[{"name": "Add", "args": {"x": 16505054784, "y": 4}, "id": "2"}],
    ),
    ToolMessage("16505054788", tool_call_id="2"),
    AIMessage(
        "The product of 317253 and 128472 plus four is 16505054788",
        name="example_assistant",
    ),
]

system = """You are bad at math but are an expert at using a calculator. 

Use past tool usage as an example of how to correctly use the tools."""
few_shot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        *examples,
        ("human", "{query}"),
    ]
)

chain = {"query": RunnablePassthrough()} | few_shot_prompt | llm_with_tools
chain.invoke("Whats 119 times 8 minus 20").tool_calls

[{'name': 'Multiply', 'args': {'a': 119, 'b': 8}, 'id': 'call_wwwz'}]

## Conclusion
Well it appears that tool calling works. Lets look at tool chains next.